# Experimento 01 — Quão previsível é `perfil_latente` só com dados de compra?

> **Status:** *Sandbox / aquecimento.* O dataset usado aqui foi disponibilizado pelo Prof. Carlos como treino; não é o dataset oficial da Ford. O objetivo deste notebook é gerar **aprendizado portátil** — pipelines, parâmetros e intuições — que serão reaplicados quando o dataset real chegar.

## Pergunta

Se a Ford tivesse apenas os dados do **momento da compra** (demografia, transação, traços inferidos no showroom) — sem ver nenhum comportamento pós-venda — **quanto ela conseguiria prever o perfil de retenção 24 meses depois?**

Se a resposta for "bastante", a Ford pode intervir *antes* do comportamento se manifestar — que é exatamente o valor central do Intelligence Hub + Action Engine do ForwardService.

## O que vamos aprender

1. **Baseline honesto** da tarefa multi-classe (4 perfis).
2. **Feature importance** — quais variáveis de compra carregam sinal de retenção. Isso vira uma *lista de campos a priorizar* quando o dataset oficial chegar.
3. **Matriz de confusão** — quais perfis a Ford conseguiria acertar fácil vs onde ela se confundiria. Com atenção especial ao perfil **esquecido**, que é estrategicamente o mais valioso.
4. **Pipeline replicável** — preprocessing + modelo num único objeto sklearn, pronto pra ser rebatizado no dataset oficial.

## Guardrails

- `perfil_latente` foi construído usando comportamento pós-venda → todas as features pós-compra estão **proibidas** aqui. Já formalizado em `data/processed/feature_dictionary.json`.
- Recall por classe > accuracy global (preferimos pegar o esquecido errando um fiel do que perdê-lo).

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Visual theme — minimal, no heavy fills (team preference)
# Tema visual — minimal, sem cores pesadas (preferência do time)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

PALETTE = ["#2E86AB", "#A23B72", "#F18F01", "#6A994E", "#6C757D"]
sns.set_palette(PALETTE)

PROFILE_ORDER = ["fiel", "economico", "esquecido", "abandono"]
RANDOM_STATE = 42

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent.parent
DATA_PROCESSED = REPO_ROOT / "data" / "processed"
FIGURES_DIR = NOTEBOOK_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(name: str) -> None:
    """Persist current figure under figures/ with tight bbox.
    Salva a figura atual em figures/ com margem apertada."""
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"{name}.png", bbox_inches="tight")


print("Data processed:", DATA_PROCESSED)
print("Figures:       ", FIGURES_DIR)

## 2. Carga — dataset processado + contrato de features

Consumimos os artefatos gerados pelo notebook 01: o parquet limpo e o `feature_dictionary.json` que define quais colunas são seguras (pré-compra) e quais são leakage (pós-compra).

In [ ]:
df = pd.read_parquet(DATA_PROCESSED / "ford_clientes_clean.parquet")
feature_dict = json.loads((DATA_PROCESSED / "feature_dictionary.json").read_text(encoding="utf-8"))

PRE_PURCHASE = feature_dict["pre_purchase"]
TARGET = "perfil_latente"

# Separate numeric and categorical for the preprocessor
# Separa numéricas e categóricas para o preprocessador
numeric_features = [c for c in PRE_PURCHASE if df[c].dtype != "object"]
categorical_features = [c for c in PRE_PURCHASE if df[c].dtype == "object"]

print(f"Pre-purchase features:  {len(PRE_PURCHASE)}")
print(f"  numeric:     {len(numeric_features)}")
print(f"  categorical: {len(categorical_features)}  {categorical_features}")
print(f"\nTarget distribution:")
print(df[TARGET].value_counts(normalize=True).round(4))

## 3. Split estratificado + preprocessing pipeline

Split 80/20 estratificado por perfil. Preprocessor encapsulado num `ColumnTransformer` — mesmo objeto será compartilhado pelos 3 modelos, garantindo comparação justa.

**Decisões:**
- Numéricas: imputação pela mediana (robusta a outliers) + padronização.
- Categóricas: imputação por `"missing"` + one-hot. `modelo_veiculo` tem 14 categorias — aceitável sem embedding.
- Sem `drop="first"` no one-hot: XGBoost lida bem com redundância; LogReg sofre um pouco, mas a comparação continua válida.

In [ ]:
X = df[PRE_PURCHASE].copy()
y_raw = df[TARGET].copy()

# Encode target to int for XGBoost — keep a mapping
# Codifica alvo pra int (XGBoost exige) mantendo mapeamento
LABEL_MAP = {name: i for i, name in enumerate(PROFILE_ORDER)}
INV_LABEL_MAP = {i: name for name, i in LABEL_MAP.items()}
y = y_raw.map(LABEL_MAP).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE,
)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), numeric_features),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical_features),
    ],
    remainder="drop",
)

## 4. Baselines — Dummy, Logistic Regression, XGBoost

Três modelos:

| Modelo | Papel |
|---|---|
| `DummyClassifier(strategy="stratified")` | Piso — o que se obtém chutando na distribuição. |
| `LogisticRegression` | Referência linear interpretável. |
| `XGBClassifier` | Modelo principal (mandatório pelo CLAUDE.md do repo). |

Métricas: **balanced accuracy** (robusta a desbalanço) + **macro F1** (trata todas as classes igual — coerente com priorizar *esquecido*).

In [ ]:
models = {
    "dummy": DummyClassifier(strategy="stratified", random_state=RANDOM_STATE),
    "logreg": LogisticRegression(max_iter=500, n_jobs=-1, random_state=RANDOM_STATE),
    "xgboost": XGBClassifier(
        objective="multi:softprob",
        num_class=len(PROFILE_ORDER),
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="mlogloss",
        tree_method="hist",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
}

results: dict[str, dict] = {}
fitted: dict[str, Pipeline] = {}
for name, clf in models.items():
    pipe = Pipeline([("pre", preprocessor), ("clf", clf)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results[name] = {
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
        "macro_f1": f1_score(y_test, pred, average="macro"),
        "pred": pred,
    }
    fitted[name] = pipe
    print(f"{name:<8}  bal_acc={results[name]['balanced_accuracy']:.4f}  macro_f1={results[name]['macro_f1']:.4f}")

In [ ]:
# Per-class classification report for the best model (XGBoost)
# Relatório por classe do melhor modelo (XGBoost)
y_pred = results["xgboost"]["pred"]
print(classification_report(
    y_test, y_pred,
    target_names=PROFILE_ORDER,
    digits=3,
))

## 5. Matriz de confusão — pra onde cada perfil é "confundido"?

Relevante pro pitch: o *esquecido* precisa ser identificável, porque é invisível aos canais de feedback. Se ele for confundido com *fiel*, perdemos a intervenção.

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[LABEL_MAP[p] for p in PROFILE_ORDER])
cm_norm = cm / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    cm_norm, annot=True, fmt=".1f", cmap="Blues",
    xticklabels=PROFILE_ORDER, yticklabels=PROFILE_ORDER,
    cbar_kws={"label": "% da linha (recall)"}, ax=ax,
)
ax.set_xlabel("Predito")
ax.set_ylabel("Real")
ax.set_title("Matriz de confusão (%) — XGBoost · apenas features pré-compra")
save_fig("05_confusion_matrix")
plt.show()

## 6. Feature importance — quais campos da compra carregam sinal?

Pega a importância gain-based do XGBoost treinado e mapeia de volta pros nomes originais (one-hot expandidas são somadas de volta à feature de origem). **Esta lista vira nossa recomendação de campos prioritários pro dataset oficial da Ford.**

In [ ]:
pipe_xgb = fitted["xgboost"]
feature_names_expanded = pipe_xgb.named_steps["pre"].get_feature_names_out()
importances_raw = pipe_xgb.named_steps["clf"].feature_importances_

# Collapse one-hot importances back to the original feature
# Agrega importâncias do one-hot de volta à feature de origem
def base_feature(expanded_name: str) -> str:
    # ColumnTransformer output looks like "num__idade" or "cat__regiao_capital"
    # Saída do ColumnTransformer tipo "num__idade" ou "cat__regiao_capital"
    stripped = expanded_name.split("__", 1)[1]
    for cat in categorical_features:
        if stripped.startswith(cat + "_"):
            return cat
    return stripped

importance_df = (
    pd.DataFrame({"expanded": feature_names_expanded, "importance": importances_raw})
    .assign(feature=lambda d: d["expanded"].map(base_feature))
    .groupby("feature", as_index=False)["importance"].sum()
    .sort_values("importance", ascending=False)
)
importance_df["pct"] = (importance_df["importance"] / importance_df["importance"].sum() * 100).round(2)
importance_df.reset_index(drop=True)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
top = importance_df.head(15).iloc[::-1]
ax.barh(top["feature"], top["importance"], color=PALETTE[0])
ax.set_xlabel("Importância agregada (gain)")
ax.set_title("Top 15 features pré-compra para prever perfil latente")
save_fig("06_feature_importance")
plt.show()

## 7. SHAP — como as features movem cada perfil

Importância gain-based responde *quais* features importam. SHAP responde *como* elas empurram a predição. Calculamos num sample de 2.000 pra manter o custo razoável (multi-classe × 500k × 23 features = inviável).

Pro pitch, o valor é: conseguimos mostrar pro avaliador/Ford uma narrativa do tipo *"entrada alta empurra o cliente pra 'fiel'; renda baixa + canal online empurra pra 'abandono'"*.

In [ ]:
import shap

# Use the preprocessed test set for SHAP — explainer works on the raw XGB model
# Usa o test set já preprocessado — o explainer opera sobre o modelo XGB puro
X_test_prep = pipe_xgb.named_steps["pre"].transform(X_test)
sample_idx = np.random.RandomState(RANDOM_STATE).choice(len(X_test_prep), size=2000, replace=False)
X_sample = X_test_prep[sample_idx]

explainer = shap.TreeExplainer(pipe_xgb.named_steps["clf"])
shap_values = explainer.shap_values(X_sample)  # shape: (n, features, classes)

# Summary plot for the "esquecido" class — strategic target
# Summary plot para a classe "esquecido" — alvo estratégico
class_idx = LABEL_MAP["esquecido"]
shap_esquecido = shap_values[:, :, class_idx] if shap_values.ndim == 3 else shap_values[class_idx]

plt.figure(figsize=(8, 6))
shap.summary_plot(
    shap_esquecido, X_sample,
    feature_names=list(feature_names_expanded),
    max_display=15, show=False,
    plot_size=(8, 6),
)
plt.title("SHAP — o que empurra um cliente para 'esquecido'")
save_fig("07_shap_esquecido")
plt.show()

In [ ]:
# Same for "fiel" — the positive end of the spectrum
# Mesmo exercício para "fiel" — extremo positivo
class_idx = LABEL_MAP["fiel"]
shap_fiel = shap_values[:, :, class_idx] if shap_values.ndim == 3 else shap_values[class_idx]

plt.figure(figsize=(8, 6))
shap.summary_plot(
    shap_fiel, X_sample,
    feature_names=list(feature_names_expanded),
    max_display=15, show=False,
    plot_size=(8, 6),
)
plt.title("SHAP — o que empurra um cliente para 'fiel'")
save_fig("07_shap_fiel")
plt.show()

## 8. Deep dive no "esquecido" — pra onde o modelo erra?

Se o esquecido for confundido com *fiel*, é grave (perdemos a intervenção). Se for confundido com *abandono*, é tolerável — o cliente seria alertado de qualquer forma. Vamos ver.

In [ ]:
esq_idx = LABEL_MAP["esquecido"]
esq_mask = y_test == esq_idx
esq_pred = pd.Series(y_pred[esq_mask]).map(INV_LABEL_MAP)

distribution = esq_pred.value_counts(normalize=True).reindex(PROFILE_ORDER).fillna(0) * 100

fig, ax = plt.subplots(figsize=(7, 3.5))
colors = [PALETTE[3] if p == "esquecido" else PALETTE[1] if p in {"fiel", "economico"} else PALETTE[2]
          for p in distribution.index]
ax.bar(distribution.index, distribution.values, color=colors)
for i, v in enumerate(distribution.values):
    ax.text(i, v, f"{v:.1f}%", ha="center", va="bottom")
ax.set_ylabel("% dos esquecidos reais")
ax.set_title("Para onde o modelo rotula os 'esquecidos' reais")
ax.set_ylim(0, max(distribution.max() * 1.15, 5))
save_fig("08_forgotten_misclassification")
plt.show()

print("\nLeitura rápida:")
print(f"  Acertou como esquecido:    {distribution['esquecido']:.1f}%  (recall)")
print(f"  Confundiu com fiel/econ:   {distribution[['fiel','economico']].sum():.1f}%  (intervenção perdida)")
print(f"  Confundiu com abandono:    {distribution['abandono']:.1f}%  (ainda alertaria)")

## 9. Learnings portáteis — o que carregamos pro dataset real

Este notebook é aquecimento, mas três coisas aqui viajam:

### a) Pipeline replicável

O objeto `pipe_xgb` é um sklearn `Pipeline(preprocessor → xgboost)` — passar um DataFrame com as mesmas colunas pré-compra e ele roda. No dataset real, a única adaptação é atualizar `PRE_PURCHASE` no `feature_dictionary.json`.

### b) Lista priorizada de campos a pedir/observar no dataset oficial

As top features deste experimento viram a **carta aos engenheiros de dados da Ford** — "destes campos abaixo precisamos ter certeza que o dataset real traz". Se o dataset oficial vier sem eles, propomos coletar.

### c) Linha de base de performance

Se o dataset real der accuracy muito melhor → é suspeito de leakage.
Se der muito pior → ou o problema é inerentemente mais difícil, ou faltam proxies que aqui estavam disponíveis (`sensibilidade_preco_inicial`, `organizacao_proxy`, `tempo_decisao_dias` — que são derivados inferidos, não dados brutos).

### Gotchas descobertos

- **`sensibilidade_preco_inicial`, `organizacao_proxy` e `tempo_decisao_dias`** são inferências, não medições diretas. Se o dataset real não tiver equivalente, precisamos propor como derivá-los (tempo entre cotação e fechamento, número de visitas, etc).
- **`modelo_veiculo` vs `categoria_veiculo`**: os dois carregam sinal parcialmente redundante. Vale testar com categoria apenas no dataset real pra ver se reduz ruído.
- **Balanced accuracy > accuracy cru** como métrica primária, porque *abandono* e *esquecido* têm churn maior e são o alvo econômico — não podem ser abafados pelo peso do *fiel*.

In [ ]:
# Persist the learnings as a JSON — this file travels to the real dataset
# Persiste os aprendizados como JSON — esse arquivo viaja pro dataset real
learnings = {
    "experiment": "01_can_we_predict_profile",
    "status": "sandbox — trained on warm-up dataset, not Ford official",
    "target": TARGET,
    "feature_scope": "pre-purchase only (no leakage)",
    "model": "XGBClassifier(n_estimators=300, max_depth=6, lr=0.1)",
    "metrics": {
        name: {k: float(v) for k, v in m.items() if k != "pred"}
        for name, m in results.items()
    },
    "top_features": importance_df.head(10).to_dict(orient="records"),
    "forgotten_recall": float(distribution["esquecido"]) / 100,
    "forgotten_lost_to_loyal": float(distribution[["fiel", "economico"]].sum()) / 100,
    "priority_fields_for_real_dataset": importance_df.head(10)["feature"].tolist(),
}
out_path = DATA_PROCESSED / "learnings_experiment_01.json"
out_path.write_text(json.dumps(learnings, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved: {out_path}")